# Tetris OpenEnv - Training with Unsloth + GRPO

Train an LLM agent to play Tetris using GRPO (Group Relative Policy Optimization).

**Environment**: Tetris on HF Spaces via OpenEnv 0.2.1
**Model**: Qwen2.5-7B-Instruct (4-bit via Unsloth)
**Training**: GRPO via TRL

Runtime: GPU T4 (free tier Colab)

In [1]:
# Cell 1: Install dependencies (remove Unsloth, use standard HF stack)
!pip uninstall unsloth unsloth-zoo -y -q
!pip install peft trl openenv-core datasets accelerate -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.9/121.9 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 633.7/633.7 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.7/251.7 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.9/201.9 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.3/152.3 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 36.4 MB/s eta 0:00:00


In [2]:
# Cell 2: Load model (standard HF stack — no Unsloth bug)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

trainable params: 80,740,352 || all params: 7,696,356,864 || trainable%: 1.0491


In [3]:
# Cell 3: Define Tetris client (standalone, no local package needed)
import json
import websockets.sync.client as ws_client

TETRIS_URL = "wss://vortexedsquirrel-tetris-env.hf.space/ws"

class TetrisClient:
    """Lightweight Tetris env client for Colab."""
    def __init__(self, url=TETRIS_URL):
        self.url = url
        self.ws = None

    def connect(self):
        self.ws = ws_client.connect(self.url, open_timeout=30)
        return self

    def _send_recv(self, msg):
        self.ws.send(json.dumps(msg))
        return json.loads(self.ws.recv(timeout=30))

    def reset(self, seed=None):
        data = {"seed": seed} if seed else {}
        resp = self._send_recv({"type": "reset", "data": data})
        d = resp["data"]
        obs = d.get("observation", d)
        return {"observation": obs, "reward": d.get("reward", 0), "done": d.get("done", False)}

    def step(self, action_str):
        resp = self._send_recv({"type": "step", "data": {"action": action_str, "metadata": {}}})
        d = resp["data"]
        obs = d.get("observation", d)
        return {"observation": obs, "reward": d.get("reward", 0), "done": d.get("done", False)}

    def close(self):
        if self.ws:
            self.ws.close()

# Quick test
client = TetrisClient().connect()
r = client.reset(seed=42)
print(f"Connected! Piece: {r['observation']['current_piece']}")
r = client.step("drop")
print(f"Step OK. Reward: {r['reward']}, Done: {r['done']}")
client.close()
print("Environment connection verified!")

Connected! Piece: L
Step OK. Reward: -7, Done: False
Environment connection verified!


In [4]:
# Cell 4: Generate training prompts — empty board, random piece, random X position
import random

# Download game engine to run locally (no network calls needed)
!wget -q -O game_engine.py https://raw.githubusercontent.com/OutOfMystic/tetris-openenv/main/src/tetris_env/server/game_engine.py

from game_engine import TetrisEnv

SYSTEM_PROMPT = """You are a Tetris AI. You see a board and must output a SEQUENCE of actions to play.
Valid actions: left, right, rotate_cw, rotate_ccw, drop, down
Output actions separated by spaces. Example: left left rotate_cw drop right drop
Strategy: position pieces to fill rows completely. Clearing multiple lines at once = bonus.
Output ONLY the action sequence, nothing else."""

VALID_ACTIONS = ["left", "right", "rotate_cw", "rotate_ccw", "drop", "down"]

def make_prompt(result):
    return f"""Board:
{result['board']}

Current piece: {result['current_piece']} (shape: {result['current_piece_shape']})
Next piece: {result['next_piece']}
Score: {result['score']} | Lines: {result['total_lines']} | Height: {result['max_height']} | Holes: {result['holes']}

Output your action sequence:"""

def generate_training_prompts(n_prompts=400):
    prompts = []
    for i in range(n_prompts):
        env = TetrisEnv(seed=i)
        result = env.reset(seed=i)

        # Random horizontal offset: move piece left or right 0-4 steps
        rng = random.Random(i)
        moves = rng.randint(0, 4)
        direction = rng.choice(["left", "right"])
        for _ in range(moves):
            env.step(direction)

        result = env._make_result(0)
        prompt = make_prompt(result)
        prompts.append({
            "prompt": [{"role": "system", "content": SYSTEM_PROMPT},
                       {"role": "user", "content": prompt}],
        })

    print(f"Generated {len(prompts)} prompts")
    from collections import Counter
    pieces = [p['prompt'][1]['content'].split('Current piece: ')[1].split(' ')[0] for p in prompts]
    print(f"Piece distribution: {dict(Counter(pieces))}")
    return prompts

train_prompts = generate_training_prompts(400)


Generated 400 prompts
Piece distribution: {'J': 58, 'O': 61, 'Z': 61, 'T': 51, 'S': 59, 'I': 69, 'L': 41}


In [5]:
# Cell 5: Create HF Dataset from prompts
from datasets import Dataset

dataset = Dataset.from_list(train_prompts)
print(f"Dataset size: {len(dataset)}")
print(f"Example prompt (user message):")
print(dataset[0]['prompt'][1]['content'][:300])
print(dataset[1]['prompt'][1]['content'][:300])
print(dataset[2]['prompt'][1]['content'][:300])

Dataset size: 400
Example prompt (user message):
Board:
+----------+
|..........|
|..........|
|..........|
|........@.|
|........@.|
|.......@@.|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
+----------+

Curren
Board:
+----------+
|..........|
|...@@.....|
|...@@.....|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
+----------+

Curren
Board:
+----------+
|.....@....|
|.....@....|
|....@@....|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
+----------+

Curren


In [6]:
# Cell 6: Reward function — plays full action sequence on local engine
# Game over = losing individual (heavy penalty)
# More lines + longer survival = better individual

def tetris_reward_func(completions, **kwargs):
    rewards = []
    for i, completion in enumerate(completions):
        text = completion[0]['content'].strip().lower() if isinstance(completion, list) else str(completion).strip().lower()

        # Parse action sequence from model output
        tokens = text.replace(",", " ").replace("\n", " ").split()
        actions = [t for t in tokens if t in VALID_ACTIONS]

        if len(actions) == 0:
            rewards.append(-20.0)  # no valid actions = very bad
            continue

        # Play full sequence on fresh engine (seed=i for reproducibility)
        env = TetrisEnv(seed=i)
        env.reset(seed=i)

        total_lines = 0
        steps_played = 0
        game_over = False

        for action in actions:
            result = env.step(action)
            steps_played += 1
            total_lines = result['total_lines']
            if result['done']:
                game_over = True
                break

        # Reward structure:
        # - Lines cleared are the main reward (with combo bonus from engine)
        # - Surviving longer = small bonus
        # - Game over = losing individual (heavy penalty)
        # - Clean board (low holes, low height) = bonus
        reward = 0.0
        reward += total_lines * 200.0          # main goal: clear lines
        reward += steps_played * 0.5           # survive longer = better
        reward -= result['holes'] * 2.0        # penalize messy boards
        reward -= result['max_height'] * 1.0   # penalize tall stacks
        if game_over:
            reward -= 50.0                     # game over = bad individual
        if len(actions) < 3:
            reward -= 10.0                     # too short sequences are lazy

        rewards.append(reward)

    return rewards

# Test with sample completions
test_completions = [
    [{"content": "left left rotate_cw drop right right drop left drop"}],
    [{"content": "drop drop drop drop drop drop drop drop"}],
    [{"content": "this is not a valid action at all"}],
    [{"content": "left"}],
]
test_rewards = tetris_reward_func(test_completions)
print(f"Test rewards: {test_rewards}")
print("Good: varied rewards → GRPO can learn from differences!")

Test rewards: [-1.5, -85.0, -20.0, -9.5]
Good: varied rewards → GRPO can learn from differences!


In [ ]:
# Cell 7: Configure and run GRPO training (A100, standard HF stack)
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="tetris-agent-grpo",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_generations=8,
    max_completion_length=256,
    learning_rate=5e-6,
    logging_steps=1,
    save_steps=50,
    bf16=True,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[tetris_reward_func],
    args=training_args,
    train_dataset=dataset,
)

print("Starting GRPO training...")
trainer.train()
print("Training complete!")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting GRPO training...


Passing `generation_config` together with generation-related arguments=({'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Step,Training Loss
1,0.095334
2,0.627495
3,1.443315
4,0.267313
5,-0.577105
6,0.669373
7,1.159806
8,0.707811
9,0.947995
10,0.293599


Step,Training Loss
1,0.095334
2,0.627495
3,1.443315
4,0.267313
5,-0.577105
6,0.669373
7,1.159806
8,0.707811
9,0.947995
10,0.293599


In [ ]:
# Cell 8: Plot reward curve
import matplotlib.pyplot as plt

logs = trainer.state.log_history
train_logs = [l for l in logs if 'loss' in l]
reward_logs = [l for l in logs if 'reward' in l or 'rewards/mean' in l]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if train_logs:
    axes[0].plot([l.get('step', i) for i, l in enumerate(train_logs)],
                 [l['loss'] for l in train_logs])
    axes[0].set_title('Training Loss')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')

reward_key = 'reward' if reward_logs and 'reward' in reward_logs[0] else 'rewards/mean'
if reward_logs:
    axes[1].plot([l.get('step', i) for i, l in enumerate(reward_logs)],
                 [l.get(reward_key, 0) for l in reward_logs])
    axes[1].set_title('Mean Reward (should go up!)')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Reward')

plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150)
plt.show()
print("Reward curve saved to reward_curve.png")

In [ ]:
# Cell 9: Test trained agent
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

client = TetrisClient().connect()
result = client.reset(seed=123)

total_reward = 0
for step_num in range(50):
    if result['done']:
        break

    prompt_text = make_prompt(result['observation'])
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt_text}
    ]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(model.device)
    outputs = model.generate(inputs, max_new_tokens=16, temperature=0.1)
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip().lower()

    # Parse action
    action = "noop"
    for valid in VALID_ACTIONS:
        if valid in response:
            action = valid
            break

    result = client.step(action)
    total_reward += result['reward']

    if step_num % 10 == 0:
        print(f"Step {step_num}: action={action}, reward={result['reward']:.1f}, score={result['observation']['score']}")

print(f"\nGame over! Total reward: {total_reward:.1f}, Score: {result['observation']['score']}, Lines: {result['observation']['total_lines']}")
print(f"\nFinal board:")
print(result['observation']['board'])
client.close()

In [ ]:
# Cell 10: Push trained model to HF Hub
model.push_to_hub("VortexedSquirrel/tetris-agent-grpo")
tokenizer.push_to_hub("VortexedSquirrel/tetris-agent-grpo")
print("Model pushed to https://huggingface.co/VortexedSquirrel/tetris-agent-grpo")